# Complete DC4 point + extended source catalog with two FITS-fitted normalizations

This notebook builds a fully fixed `astromodels` catalog from every
simulation component listed for the DC4 mock dataset.  It starts with
the reviewed continuum models from `Source_catalog_generator_comp_fit_norm.ipynb`,
then reads the remaining MEGAlib `.source` definitions from the cloned
`cosi-sim` source library.  Point, Gaussian-extended, and tabulated
three-dimensional sky models are retained.

Two independent catalogs are written:

1. **full** — every amplitude is calibrated to its complete native
   three-month FITS file and the full orientation;
2. **NGC 4151 GTI** — every amplitude is calibrated after the NGC 4151
   60-degree pointing/occultation GTI is applied to both events and
   orientation.

No light-curve-weighted response is used.  Consequently the fitted
amplitudes of flares and bursts are time-averaged over the stated full
or GTI exposure and their light curves must not be applied again.

The challenge documentation calls the mock dataset a 64-source
simulation, while its current download manifest contains 71 standalone
source simulation files because several physical sources are split into
line/continuum or low/high-state files.  This notebook follows the
manifest: all 71 files are represented explicitly and no dark-matter
files (which the manifest says were not included) are added.

In [1]:
from copy import deepcopy
import gc
import gzip
from pathlib import Path
import re

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astromodels import (
    Asymm_Gaussian_on_sphere,
    Band,
    Constant,
    Cutoff_powerlaw,
    ExtendedSource,
    Gaussian,
    Gaussian_on_sphere,
    Line,
    LinearPolarization,
    Log_parabola,
    Model,
    PointSource,
    Powerlaw,
    SpectralComponent,
    load_model,
)
from histpy import Axis, Axes, HealpixAxis, Histogram

from cosipy.data_io import BinnedData, EmCDSBinnedData
from cosipy.event_selection import GoodTimeInterval
from cosipy.response import (
    BinnedInstrumentResponse,
    BinnedThreeMLExtendedSourceResponse,
    BinnedThreeMLModelFolding,
    BinnedThreeMLPointSourceResponse,
    ExtendedSourceResponse,
)
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat

%matplotlib inline

00:08:46 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=500211;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=346999;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=666072;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=268267;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

         WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=932526;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=89643;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

00:08:46 INFO      Starting 3ML!                                                                     ]8;id=592256;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=9333;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=556719;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=879039;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=938782;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=806973;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

         WARNING   No fermitools installed                                              ]8;id=27861;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=596696;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable OMP_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=97430;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=71385;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=222092;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=230799;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=922041;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=605593;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

## Locate the injected spectral tables

The tables are read directly from the local `cositools/cosi-sim` clone at `/Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library`. The helper searches recursively below its `DC3/` and `DC4/` directories.

The catalog uses the following injected tables where a regular astromodels function is not a faithful representation:

- `cygX1_hard_0.1-0.4_spec.dat`
- `cygX1_hard_0.4-10_spec.dat`
- `GRS1758_spec.dat`
- `Crab_Ne_spec.dat`, `Crab_P1_spec.dat`, `Crab_Brg_spec.dat`, and `Crab_P2_spec.dat`
- `4C71p07_spectrum.dat`
- `4C_21_spectrum_noflare.dat` and `4C_21_spectrum_flare.dat`
- `MAXIJ1820_0.1-0.4_spec.dat` and `MAXIJ1820_0.4-10_spec.dat`
- `MAXIJ1348_0.1-0.4_spec.dat` and `MAXIJ1348_0.4-10_spec.dat`
- `3C454p3_low_spectrum.dat` and `3C454p3_high_spectrum.dat`
- `cygX3_transition_spec.dat`, `PSRB1259_spec.dat`, `LS5039_spec.dat`, and `J1846_spec.dat`
- `nova_co_continuum_spec.dat`

The individual simulated FITS source files in `DC4_Files/DC4_Sources` are event-data products and cannot replace these spectral `.dat` files.

In [2]:
SOURCE_DATA_ROOT = Path(
    "/Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library"
)


def find_source_file(challenge, filename):
    """Find one source-library file below DC3/ or DC4/."""
    search_root = SOURCE_DATA_ROOT / challenge
    matches = sorted(search_root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} below {search_root}. "
            "The simulated FITS source files are not spectral tables; copy the "
            "corresponding .dat file from the cosi-sim source library into this "
            "DC3/DC4 directory."
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Found more than one copy of {filename!r}: {matches}. "
            "Keep one copy or set the desired path explicitly."
        )

    return matches[0].resolve()


cygx1_low_path = find_source_file("DC3", "cygX1_hard_0.1-0.4_spec.dat")
cygx1_high_path = find_source_file("DC3", "cygX1_hard_0.4-10_spec.dat")
grs1758_path = find_source_file("DC3", "GRS1758_spec.dat")
crab_nebula_path = find_source_file("DC4", "Crab_Ne_spec.dat")
crab_p1_path = find_source_file("DC4", "Crab_P1_spec.dat")
crab_bridge_path = find_source_file("DC4", "Crab_Brg_spec.dat")
crab_p2_path = find_source_file("DC4", "Crab_P2_spec.dat")
four_c_71p07_path = find_source_file("DC4", "4C71p07_spectrum.dat")
four_c_21p35_noflare_path = find_source_file(
    "DC3", "4C_21_spectrum_noflare.dat"
)
four_c_21p35_flare_path = find_source_file(
    "DC3", "4C_21_spectrum_flare.dat"
)
maxi_j1820_low_path = find_source_file("DC3", "MAXIJ1820_0.1-0.4_spec.dat")
maxi_j1820_high_path = find_source_file("DC3", "MAXIJ1820_0.4-10_spec.dat")
maxi_j1348_low_path = find_source_file("DC3", "MAXIJ1348_0.1-0.4_spec.dat")
maxi_j1348_high_path = find_source_file("DC3", "MAXIJ1348_0.4-10_spec.dat")
three_c_454p3_low_path = find_source_file("DC4", "3C454p3_low_spectrum.dat")
three_c_454p3_high_path = find_source_file("DC4", "3C454p3_high_spectrum.dat")
cyg_x3_path = find_source_file("DC4", "cygX3_transition_spec.dat")
psr_b1259_path = find_source_file("DC3", "PSRB1259_spec.dat")
ls_5039_path = find_source_file("DC3", "LS5039_spec.dat")
psr_j1846_path = find_source_file("DC3", "J1846_spec.dat")
co_nova_continuum_path = find_source_file("DC3", "nova_co_continuum_spec.dat")

print("Cyg X-1 low-energy table:", cygx1_low_path)
print("Cyg X-1 high-energy table:", cygx1_high_path)
print("GRS 1758 table:", grs1758_path)
print("Crab tables:", crab_nebula_path, crab_p1_path, crab_bridge_path, crab_p2_path)
print("4C 71.07 table:", four_c_71p07_path)
print("4C 21.35 tables:", four_c_21p35_noflare_path, four_c_21p35_flare_path)
print("MAXI J1820 tables:", maxi_j1820_low_path, maxi_j1820_high_path)
print("MAXI J1348 tables:", maxi_j1348_low_path, maxi_j1348_high_path)
print("3C 454.3 tables:", three_c_454p3_low_path, three_c_454p3_high_path)
print("Cyg X-3 table:", cyg_x3_path)
print("PSR B1259 table:", psr_b1259_path)
print("LS 5039 table:", ls_5039_path)
print("PSR J1846 table:", psr_j1846_path)
print("CO nova continuum table:", co_nova_continuum_path)

Cyg X-1 low-energy table: /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC3/sources/Galactic/cygX1_hard/cygX1_hard_0.1-0.4_spec.dat
Cyg X-1 high-energy table: /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC3/sources/Galactic/cygX1_hard/cygX1_hard_0.4-10_spec.dat
GRS 1758 table: /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC3/sources/Galactic/GRS1758/GRS1758_spec.dat
Crab tables: /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC4/sources/Galactic/Crab/Crab_Ne_spec.dat /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC4/sources/Galactic/Crab/Crab_P1_spec.dat /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC4/sources/Galactic/Crab/Crab_Brg_spec.dat /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC4/sources/Galactic/Crab/Crab_P2_spec.dat
4C 71.07 table: /Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library/DC4/sources/Extragalactic/4C71p07/4C71p07_spectrum.dat
4C 21.35 tables: /Users/parshadkp/Sof

## Table-spectrum helper

`SpecFromDat` preserves the tabulated shape and provides a normalization parameter. Its current normalization uses a simple bin-width sum, so the helper below performs a numerical correction such that the model integral from 100 keV to 10 MeV equals the injected source-file flux. Absolute paths are stored in the YAML so the model can be reloaded from another working directory. The normalization is fixed before serialization.

In [3]:
ENERGY_MIN = 100.0
ENERGY_MAX = 10_000.0
PHOTON_FLUX_UNIT = 1 / (u.cm**2 * u.s)
DIFFERENTIAL_FLUX_UNIT = 1 / (u.keV * u.cm**2 * u.s)


def table_spectrum(dat_path, injected_photon_flux):
    """Create a SpecFromDat component normalized to an integrated flux."""
    spectrum = SpecFromDat(
        K=float(injected_photon_flux),
        dat=Path(dat_path).resolve(),
    )

    energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
    current_integral = np.trapezoid(spectrum(energy), energy)

    if not np.isfinite(current_integral) or current_integral <= 0:
        raise ValueError(f"Invalid spectrum integral for {dat_path}: {current_integral}")

    spectrum.K.value *= float(injected_photon_flux) / current_integral
    return spectrum


def normalize_analytic_components(components, injected_photon_flux):
    """Scale analytic component amplitudes to the injected integrated flux."""
    energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
    current_integral = np.trapezoid(
        sum(component(energy) for component in components),
        energy,
    )

    if not np.isfinite(current_integral) or current_integral <= 0:
        raise ValueError(f"Invalid analytic spectrum integral: {current_integral}")

    scale = float(injected_photon_flux) / current_integral
    for component in components:
        component.K.value *= scale


def set_normalization_step(parameter):
    """Use a scale-appropriate initial optimizer step."""
    parameter.delta = max(abs(parameter.value) * 0.05, 1e-12)

## 1. Cyg X-1 hard state

The injected hard-state spectrum is an `eqpair` calculation divided into two energy ranges for the polarization simulation. Native astromodels does not provide this `eqpair` model, so the two injected tables are retained and their relative normalization is fixed.

In [4]:
cygx1_low_flux = 0.04243172227636306
cygx1_high_flux = 0.003371822180706115

cygx1_low = table_spectrum(cygx1_low_path, cygx1_low_flux)
cygx1_high = table_spectrum(cygx1_high_path, cygx1_high_flux)
cygx1_spectrum = cygx1_low + cygx1_high

cygx1 = PointSource(
    "cyg_x1_hard",
    l=71.33496,
    b=3.066917,
    spectral_shape=cygx1_spectrum,
)

## 2. Crab

The DC4 Crab is retained as four distinct astromodels spectral components:
nebula, peak 1, bridge, and peak 2. Each uses its exact injected table and
integrated flux. The nebula has 40% polarization at 160° and the three pulsar
components have 20% polarization at 145°, in Galactic/IAU coordinates.

The pulsar light curves repeat every 0.0333924123 s and are normalized to unit
mean by MEGAlib. Each 15 s spacecraft interval therefore averages over about
449 pulse periods, so the time-integrated response uses their mean spectra.
The catalog preserves the injected polarization metadata, but exact
polarized folding additionally requires a detector response with a `Pol` axis.
The current continuum response lacks that axis, so the fitting notebook reports
and uses an in-memory zero-polarization approximation.

Although `Crab.source` contains `EarthOccultation false`, transferring that flag
to the current binned response overpredicts the selected Crab counts by almost
an order of magnitude. The spectral fit therefore retains the response's
standard source-visibility treatment, which agrees with the selected DC4 Crab
events.


In [5]:
crab_nebula = table_spectrum(crab_nebula_path, 0.033197515)
crab_p1 = table_spectrum(crab_p1_path, 0.002380673782066656)
crab_bridge = table_spectrum(crab_bridge_path, 0.0006280494007254569)
crab_p2 = table_spectrum(crab_p2_path, 0.0032012384701172636)

crab_components = [
    SpectralComponent(
        "nebula",
        crab_nebula,
        LinearPolarization(40.0, 160.0),
    ),
    SpectralComponent(
        "peak1",
        crab_p1,
        LinearPolarization(20.0, 145.0),
    ),
    SpectralComponent(
        "bridge",
        crab_bridge,
        LinearPolarization(20.0, 145.0),
    ),
    SpectralComponent(
        "peak2",
        crab_p2,
        LinearPolarization(20.0, 145.0),
    ),
]

crab = PointSource(
    "crab",
    l=184.5575,
    b=-5.78434,
    components=crab_components,
)

for component in crab.components.values():
    component.polarization.degree.fix = True
    component.polarization.angle.fix = True


## 3. 1E 1740.7−2942

The injected `compow` table is accurately represented in the COSI energy range by a cutoff power law plus a high-energy power-law tail. Their relative normalization is linked, and the remaining overall amplitude is fixed before the catalog is saved.

In [6]:
one_e_thermal = Cutoff_powerlaw()
one_e_thermal.K.value = 7.871196815455384e-4
one_e_thermal.piv.value = 300.0
one_e_thermal.index.value = 0.4379077571700285
one_e_thermal.xc.value = 31.912756594453995

one_e_tail = Powerlaw()
one_e_tail.K.value = 3.929675541578979e-6
one_e_tail.piv.value = 300.0
one_e_tail.index.value = -1.8998638815533166

for spectrum in (one_e_thermal, one_e_tail):
    spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
    spectrum.piv.unit = u.keV

one_e_thermal.xc.unit = u.keV

for parameter in (
    one_e_thermal.piv,
    one_e_thermal.index,
    one_e_thermal.xc,
    one_e_tail.piv,
    one_e_tail.index,
):
    parameter.fix = True

one_e_spectrum = one_e_thermal + one_e_tail

one_e_1740 = PointSource(
    "one_e_1740_compow",
    l=359.11596,
    b=-0.10575,
    spectral_shape=one_e_spectrum,
)

## 4. GRS 1758−258

The injected spectrum is a thermal-Comptonization model. The original table is retained rather than replacing its curvature with a simple power law.

In [7]:
grs1758_flux = 0.003495
grs1758_spectrum = table_spectrum(grs1758_path, grs1758_flux)

grs1758 = PointSource(
    "grs_1758_258",
    l=4.50780,
    b=-1.36106,
    spectral_shape=grs1758_spectrum,
)

## 5. Cen A

Cen A is injected as a power law with photon index −1.732 and integrated 100 keV–10 MeV flux 0.00197 ph cm$^{-2}$ s$^{-1}$.

In [8]:
cena_spectrum = Powerlaw()
cena_spectrum.K.value = 2.227338148135328e-6
cena_spectrum.piv.value = 300.0
cena_spectrum.index.value = -1.732

cena_spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
cena_spectrum.piv.unit = u.keV
cena_spectrum.piv.fix = True
cena_spectrum.index.fix = True
cena = PointSource(
    "cena",
    l=309.516,
    b=19.417,
    spectral_shape=cena_spectrum,
)

## 6. 4C 71.07

4C 71.07 retains its injected tabulated spectrum and is treated as steady.

In [9]:
four_c_71p07_spectrum = table_spectrum(four_c_71p07_path, 0.00119)
four_c_71p07 = PointSource(
    "four_c_71p07",
    l=143.540759,
    b=34.425671,
    spectral_shape=four_c_71p07_spectrum,
)

## 7. 4C 21.35

The non-flare and flare spectra are separate variable catalog entries. Each is fitted with the ordinary NGC 4151 GTI response, so its frozen amplitude is the component's time-averaged contribution inside this GTI.

In [10]:
four_c_21p35_noflare_spectrum = table_spectrum(
    four_c_21p35_noflare_path,
    0.0005535,
)
four_c_21p35_noflare = PointSource(
    "four_c_21p35_noflare",
    l=255.073637,
    b=81.659766,
    spectral_shape=four_c_21p35_noflare_spectrum,
)

four_c_21p35_flare_spectrum = table_spectrum(
    four_c_21p35_flare_path,
    0.017293201590129727,
)
four_c_21p35_flare = PointSource(
    "four_c_21p35_flare",
    l=255.073637,
    b=81.659766,
    spectral_shape=four_c_21p35_flare_spectrum,
)

## 8. MAXI J1820

The two injected tabulated components share one outburst light curve and one linked catalog amplitude.

In [11]:
maxi_j1820_low = table_spectrum(
    maxi_j1820_low_path,
    0.13820979415525897,
)
maxi_j1820_high = table_spectrum(
    maxi_j1820_high_path,
    0.005963517352694539,
)
maxi_j1820 = PointSource(
    "maxi_j1820",
    l=35.8535,
    b=10.15915,
    spectral_shape=maxi_j1820_low + maxi_j1820_high,
)

## 9. MAXI J1348−630

The two injected tabulated components share one outburst light curve and one linked catalog amplitude.

In [12]:
maxi_j1348_low = table_spectrum(
    maxi_j1348_low_path,
    0.08633295828868433,
)
maxi_j1348_high = table_spectrum(
    maxi_j1348_high_path,
    0.0023113717669982913,
)
maxi_j1348 = PointSource(
    "maxi_j1348",
    l=309.26389732,
    b=-1.10328,
    spectral_shape=maxi_j1348_low + maxi_j1348_high,
)

## 10. 3C 454.3

The low and high states are separate variable catalog entries. Their normalizations are fitted independently from their native FITS events inside the NGC 4151 GTI.

In [13]:
three_c_454p3_low = PointSource(
    "three_c_454p3_low",
    l=86.111069,
    b=-38.183817,
    spectral_shape=table_spectrum(three_c_454p3_low_path, 0.00029),
)
three_c_454p3_high = PointSource(
    "three_c_454p3_high",
    l=86.111069,
    b=-38.183817,
    spectral_shape=table_spectrum(three_c_454p3_high_path, 0.00624),
)

## 11. NGC 1068

NGC 1068 uses its injected cutoff-power-law plus high-energy power-law tail. The components are normalized to the injected 100 keV–10 MeV photon flux and linked at their injected ratio.

In [14]:
ngc1068_thermal = Cutoff_powerlaw()
ngc1068_thermal.K.value = 0.308 * 200.0**-1.92
ngc1068_thermal.piv.value = 200.0
ngc1068_thermal.index.value = -1.92
ngc1068_thermal.xc.value = 200.0
ngc1068_tail = Powerlaw()
ngc1068_tail.K.value = 91.18 * 200.0**-3.8
ngc1068_tail.piv.value = 200.0
ngc1068_tail.index.value = -3.8

for component in (ngc1068_thermal, ngc1068_tail):
    component.K.unit = DIFFERENTIAL_FLUX_UNIT
    component.piv.unit = u.keV
    component.piv.fix = True
    component.index.fix = True

ngc1068_thermal.xc.unit = u.keV
ngc1068_thermal.xc.fix = True
normalize_analytic_components((ngc1068_thermal, ngc1068_tail), 0.00161)

ngc1068 = PointSource(
    "ngc1068",
    l=172.103584,
    b=-51.933771,
    spectral_shape=ngc1068_thermal + ngc1068_tail,
)

## 12. NGC 4151

This is the updated mock-data injection from `AGN_Corona_DC4.ipynb`: a cutoff power law with `K = 0.15` at 1 keV, index −1.75, and cutoff 200 keV, plus an index −3.8 power-law tail whose differential flux at 200 keV is 30% of the thermal flux. The implementation is pivoted at 200 keV without changing the injected spectrum.

In [15]:
ngc4151_thermal = Cutoff_powerlaw()
ngc4151_thermal.K.value = 0.15 * 200.0**-1.75
ngc4151_thermal.piv.value = 200.0
ngc4151_thermal.index.value = -1.75
ngc4151_thermal.xc.value = 200.0
ngc4151_tail = Powerlaw()
ngc4151_tail_fraction_at_200kev = 0.30
ngc4151_tail.K.value = (
    ngc4151_tail_fraction_at_200kev
    * ngc4151_thermal.evaluate_at(200.0)
)
ngc4151_tail.piv.value = 200.0
ngc4151_tail.index.value = -3.8

for component in (ngc4151_thermal, ngc4151_tail):
    component.K.unit = DIFFERENTIAL_FLUX_UNIT
    component.piv.unit = u.keV
    component.piv.fix = True
    component.index.fix = True

ngc4151_thermal.xc.unit = u.keV
ngc4151_thermal.xc.fix = True
assert np.isclose(
    ngc4151_tail.evaluate_at(200.0)
    / ngc4151_thermal.evaluate_at(200.0),
    ngc4151_tail_fraction_at_200kev,
)

ngc4151_energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
ngc4151_injected_flux = np.trapezoid(
    (ngc4151_thermal + ngc4151_tail)(ngc4151_energy),
    ngc4151_energy,
)
assert np.isclose(ngc4151_injected_flux, 0.0025171894, rtol=1e-5)

ngc4151 = PointSource(
    "ngc4151",
    l=155.077404,
    b=75.063170,
    spectral_shape=ngc4151_thermal + ngc4151_tail,
)

## 13. Cyg X-3

The injected transition-state spectrum is retained as a tabulated model.

In [16]:
cyg_x3 = PointSource(
    "cyg_x3",
    l=79.84549,
    b=0.70006,
    spectral_shape=table_spectrum(cyg_x3_path, 0.001331405),
)

## 14. PSR B1259−63

The injected table defines the spectral shape. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, absorbing the light-curve average for this selection.

In [17]:
psr_b1259 = PointSource(
    "psr_b1259",
    l=304.18358257,
    b=-0.99158457,
    spectral_shape=table_spectrum(psr_b1259_path, 0.00061253),
)

## 15. 1RXS J170849.0−400901

The injected magnetar continuum is represented directly with a fixed astromodels `Log_parabola`.

In [18]:
one_rxs_j170849_spectrum = Log_parabola()
one_rxs_j170849_spectrum.K.value = 1.68e-6
one_rxs_j170849_spectrum.piv.value = 143.276
one_rxs_j170849_spectrum.alpha.value = -1.637
one_rxs_j170849_spectrum.beta.value = 0.261
one_rxs_j170849_spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
one_rxs_j170849_spectrum.piv.unit = u.keV
one_rxs_j170849_spectrum.piv.fix = True
one_rxs_j170849_spectrum.alpha.fix = True
one_rxs_j170849_spectrum.beta.fix = True
normalize_analytic_components((one_rxs_j170849_spectrum,), 0.00033)
one_rxs_j170849 = PointSource(
    "one_rxs_j170849",
    l=346.47938142,
    b=0.03838608,
    spectral_shape=one_rxs_j170849_spectrum,
)

## 16. Generic magnetar

The generic magnetar uses the injected fixed `Log_parabola`, with twice the curvature parameter of the 1RXS J170849.0−400901 model.

In [19]:
generic_magnetar_spectrum = Log_parabola()
generic_magnetar_spectrum.K.value = 1.68e-6
generic_magnetar_spectrum.piv.value = 143.276
generic_magnetar_spectrum.alpha.value = -1.637
generic_magnetar_spectrum.beta.value = 0.522
generic_magnetar_spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
generic_magnetar_spectrum.piv.unit = u.keV
generic_magnetar_spectrum.piv.fix = True
generic_magnetar_spectrum.alpha.fix = True
generic_magnetar_spectrum.beta.fix = True
normalize_analytic_components((generic_magnetar_spectrum,), 0.00033)

generic_magnetar = PointSource(
    "generic_magnetar",
    l=250.0,
    b=0.03838608,
    spectral_shape=generic_magnetar_spectrum,
)

## 17. LS 5039

The injected table defines the spectral shape. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, absorbing the periodic average for this selection.

In [20]:
ls_5039 = PointSource(
    "ls_5039",
    l=16.88158741,
    b=-1.28921165,
    spectral_shape=table_spectrum(ls_5039_path, 0.0003006),
)

## 18. PSR J1846−0258

The injected table defines the spectral shape. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, absorbing the phase average for this selection.

In [21]:
psr_j1846 = PointSource(
    "psr_j1846",
    l=29.71195,
    b=-0.24012,
    spectral_shape=table_spectrum(psr_j1846_path, 0.000152392),
)

## 19. CO nova continuum

Only the continuum component is included here. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, so the frozen amplitude absorbs the short nova duration in this selection.

In [22]:
co_nova_continuum = PointSource(
    "co_nova_continuum",
    l=310.9847,
    b=2.7256,
    spectral_shape=table_spectrum(co_nova_continuum_path, 0.122071),
)

## Parse the remaining DC4 MEGAlib source definitions

The helper below supports every beam/spectrum syntax used by the
included manifest: point sources, circular/asymmetric Gaussian sky
morphologies, Gaussian lines, Band, Comptonized, power-law and file
spectra, plus `FarFieldNormalizedEnergyBeamFluxFunction` maps.

The latter are genuinely energy-dependent spatial templates.  They are
converted once to the HEALPix FITS layout consumed by
`GalpropHealpixModel`; replacing them with a point source or a separable
one-dimensional `.dat` spectrum would lose their injected morphology.

In [23]:
SPATIAL_TEMPLATE_DIRECTORY = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Radio_Quiet_AGN/DC4_Files/DC4_Sources/DC4_Spatial_Templates"
).resolve()
# Match the spatial-template grid to the cached extended-response grid.
# Build each MEGAlib map at higher resolution first, then area-average it
# onto NSIDE 8 so compact sources cannot fall between coarse pixel centers.
SPATIAL_TEMPLATE_NSIDE = 8
SPATIAL_TEMPLATE_OVERSAMPLE_NSIDE = 64
SPATIAL_TEMPLATE_FORMAT_TAG = "perMeV_v3"
OVERWRITE_SPATIAL_TEMPLATES = False


def safe_name(value):
    name = re.sub(r"[^A-Za-z0-9_]", "_", str(value)).strip("_").lower()
    if not name or name[0].isdigit():
        name = f"src_{name}"
    return name


def find_source_definition(challenge, source_stem):
    matches = sorted(
        (SOURCE_DATA_ROOT / challenge / "sources").rglob(
            f"{source_stem}.source"
        )
    )
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {source_stem}.source below {challenge}; "
            f"found {matches}"
        )
    return matches[0].resolve()


def resolve_source_asset(source_definition, filename):
    path = source_definition.parent / filename
    candidates = [path, Path(f"{path}.gz")]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not resolve {filename!r} next to {source_definition}"
    )


def parse_megalib_blocks(source_definition):
    # Return one property dictionary per DataChallenge.Source block.
    blocks = []
    by_megalib_name = {}
    for raw_line in source_definition.read_text().splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        tokens = line.split()
        if tokens[:1] == ["DataChallenge.Source"]:
            megalib_name = tokens[1]
            block = {"_name": megalib_name}
            blocks.append(block)
            by_megalib_name[megalib_name] = block
            continue
        if "." not in tokens[0]:
            continue
        megalib_name, prop = tokens[0].split(".", 1)
        if megalib_name in by_megalib_name:
            by_megalib_name[megalib_name][prop] = tokens[1:]
    if not blocks:
        raise ValueError(f"No DataChallenge.Source blocks in {source_definition}")
    return blocks


def normalization_parameter(shape):
    candidates = [
        parameter
        for parameter in shape.parameters.values()
        if parameter.is_normalization
    ]
    if not candidates:
        # astromodels marks the Gaussian-line amplitude F as a prior
        # parameter rather than a source normalization.  It is still the
        # multiplicative integrated line flux for this use.
        candidates = [
            shape.parameters[name]
            for name in ("F", "K", "k")
            if name in shape.parameters
        ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one normalization in {shape.name}; found {candidates}"
        )
    return candidates[0]


def normalize_shape_in_band(shape, photon_flux):
    energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
    integral = float(np.trapezoid(shape(energy), energy))
    if not np.isfinite(integral) or integral <= 0:
        raise ValueError(f"Invalid integral {integral} for {shape}")
    normalization_parameter(shape).value *= float(photon_flux) / integral
    return shape


def make_parsed_spectrum(block, source_definition):
    tokens = block["Spectrum"]
    kind = tokens[0].lower()
    photon_flux = float(block["Flux"][0])

    if kind == "file":
        return table_spectrum(
            resolve_source_asset(source_definition, tokens[1]),
            photon_flux,
        )

    if kind == "gaussian":
        shape = Gaussian()
        shape.F.value = 1.0
        shape.mu.value = float(tokens[1])
        shape.sigma.value = float(tokens[2])
        return normalize_shape_in_band(shape, photon_flux)

    if kind == "band":
        shape = Band()
        shape.K.value = 1.0e-5
        shape.piv.value = 100.0
        shape.alpha.value = float(tokens[3])
        shape.beta.value = float(tokens[4])
        shape.xp.value = float(tokens[5])
        return normalize_shape_in_band(shape, photon_flux)

    if kind == "comptonized":
        alpha = float(tokens[3])
        epeak = float(tokens[4])
        shape = Cutoff_powerlaw()
        shape.K.value = 1.0e-5
        shape.piv.value = 100.0
        shape.index.value = alpha
        shape.xc.value = epeak / (2.0 + alpha)
        return normalize_shape_in_band(shape, photon_flux)

    if kind == "powerlaw":
        shape = Powerlaw()
        shape.K.value = 1.0e-5
        shape.piv.value = 100.0
        shape.index.value = -float(tokens[3])
        return normalize_shape_in_band(shape, photon_flux)

    raise NotImplementedError(
        f"Unsupported spectrum {block['Spectrum']} in {source_definition}"
    )


def make_polarization(block):
    tokens = block.get("Polarization")
    if tokens is None:
        return None
    # MEGAlib stores degree as a fraction; astromodels uses percent.
    polarization = LinearPolarization(
        100.0 * float(tokens[-2]),
        float(tokens[-1]),
    )
    polarization.degree.fix = True
    polarization.angle.fix = True
    return polarization

In [24]:
def read_normalized_energy_beam(path):
    # Read a MEGAlib 3D Functions PA/TA/EA/AP table.
    opener = gzip.open if Path(path).suffix == ".gz" else open
    pa = ta = ea = None
    entries = []
    with opener(path, "rt") as stream:
        for raw_line in stream:
            tokens = raw_line.split()
            if not tokens:
                continue
            if tokens[0] == "PA":
                pa = np.asarray(tokens[1:], dtype=float)
            elif tokens[0] == "TA":
                ta = np.asarray(tokens[1:], dtype=float)
            elif tokens[0] == "EA":
                ea = np.asarray(tokens[1:], dtype=float)
            elif tokens[0] == "AP":
                entries.append(
                    (
                        int(tokens[1]),
                        int(tokens[2]),
                        int(tokens[3]),
                        float(tokens[4]),
                    )
                )
    if pa is None or ta is None or ea is None:
        raise ValueError(f"Missing PA/TA/EA axes in {path}")
    values = np.zeros((pa.size, ta.size, ea.size), dtype=np.float32)
    for i_pa, i_ta, i_ea, value in entries:
        values[i_pa, i_ta, i_ea] = value
    return pa, ta, ea, values


def nearest_axis_indices(axis, values):
    order = np.argsort(axis)
    sorted_axis = axis[order]
    insertion = np.searchsorted(sorted_axis, values)
    insertion = np.clip(insertion, 1, sorted_axis.size - 1)
    left = sorted_axis[insertion - 1]
    right = sorted_axis[insertion]
    choose_right = np.abs(values - right) < np.abs(values - left)
    nearest = insertion - 1 + choose_right.astype(int)
    return order[nearest]


def megalib_map_to_galprop_fits(
    input_path,
    output_path,
    nside=SPATIAL_TEMPLATE_NSIDE,
    overwrite=OVERWRITE_SPATIAL_TEMPLATES,
):
    # Convert the injected coupled spatial/spectral table to HEALPix.
    output_path = Path(output_path).resolve()
    if output_path.exists() and not overwrite:
        return output_path

    pa, ta, energy_kev, values = read_normalized_energy_beam(input_path)
    sampling_nside = max(int(nside), int(SPATIAL_TEMPLATE_OVERSAMPLE_NSIDE))
    npix = hp.nside2npix(sampling_nside)
    longitude, latitude = hp.pix2ang(
        sampling_nside,
        np.arange(npix),
        lonlat=True,
    )
    longitude = ((longitude + 180.0) % 360.0) - 180.0
    theta = 90.0 - latitude
    i_pa = nearest_axis_indices(pa, longitude)
    i_ta = nearest_axis_indices(ta, theta)
    sampled_values = values[i_pa, i_ta, :]

    if sampling_nside == int(nside):
        healpix_values = sampled_values
    else:
        # The table stores intensity, so power=0 preserves the area-weighted
        # mean intensity and therefore the total flux after pixel-area folding.
        healpix_values = np.column_stack(
            [
                hp.ud_grade(
                    sampled_values[:, energy_index],
                    nside_out=int(nside),
                    order_in="RING",
                    order_out="RING",
                    power=0,
                )
                for energy_index in range(sampled_values.shape[1])
            ]
        ).astype(np.float32, copy=False)

    # GalpropHealpixModel interpolates at response-bin edges.  Add zero
    # anchors outside the simulated band to avoid extrapolated line tails.
    energy_mev = energy_kev / 1000.0
    # MEGAlib energies and differential intensities are per keV, whereas
    # GalpropHealpixModel expects energy in MeV and intensity per MeV.
    # Preserve flux under the coordinate change: f_MeV = 1000 * f_keV.
    healpix_values *= 1000.0
    if np.any(np.diff(energy_mev) <= 0.0):
        raise ValueError(f"Energy axis is not strictly increasing in {input_path}")
    epsilon = 1.0e-6
    lower_inner = energy_mev[0] * (1.0 - epsilon)
    upper_inner = energy_mev[-1] * (1.0 + epsilon)
    lower_outer = min(ENERGY_MIN / 1000.0, lower_inner)
    upper_outer = max(ENERGY_MAX / 1000.0, upper_inner)
    if lower_outer >= lower_inner:
        lower_outer = lower_inner * (1.0 - epsilon)
    if upper_outer <= upper_inner:
        upper_outer = upper_inner * (1.0 + epsilon)
    energy_mev = np.concatenate(
        ([lower_outer, lower_inner], energy_mev, [upper_inner, upper_outer])
    )
    zero_columns = np.zeros((healpix_values.shape[0], 2), dtype=np.float32)
    healpix_values = np.column_stack(
        (
            zero_columns,
            healpix_values,
            zero_columns,
        )
    )

    skymap_columns = [
        fits.Column(
            name=f"ENERGY{index:03d}",
            format="E",
            unit="ph cm-2 s-1 sr-1 MeV-1",
            array=healpix_values[:, index],
        )
        for index in range(energy_mev.size)
    ]
    skymap = fits.BinTableHDU.from_columns(
        skymap_columns,
        name="SKYMAP",
    )
    energies = fits.BinTableHDU.from_columns(
        [
            fits.Column(
                name="ENERGY",
                format="E",
                unit="MeV",
                array=energy_mev,
            )
        ],
        name="ENERGIES",
    )
    skymap.header["NSIDE"] = nside
    skymap.header["ORDERING"] = "RING"
    skymap.header["COORDSYS"] = "GAL"
    skymap.header["MAPVERS"] = SPATIAL_TEMPLATE_FORMAT_TAG
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fits.HDUList([fits.PrimaryHDU(), skymap, energies]).writeto(
        output_path,
        overwrite=overwrite,
    )
    return output_path


def make_map_source(unit_key, block, source_definition):
    map_path = resolve_source_asset(source_definition, block["Beam"][1])
    template_path = megalib_map_to_galprop_fits(
        map_path,
        SPATIAL_TEMPLATE_DIRECTORY
        / (
            f"{unit_key}_nside{SPATIAL_TEMPLATE_NSIDE}_"
            f"{SPATIAL_TEMPLATE_FORMAT_TAG}.fits"
        ),
    )
    spatial = GalpropHealpixModel()
    # Defer loading large templates until the response is folded.
    spatial._fitsfile = str(template_path)
    spatial._file_loaded = False
    spatial.K.value = 1.0
    # A 3D astromodels spatial template automatically receives the fixed
    # spectrum.main.Constant placeholder required by COSI folding.
    return ExtendedSource(unit_key, spatial_shape=spatial)

In [25]:
def parse_megalib_source(unit_key, source_definition):
    # Build one astromodels source from one manifest simulation file.
    blocks = parse_megalib_blocks(source_definition)
    beam_kinds = {block["Beam"][0] for block in blocks}
    if beam_kinds == {"FarFieldNormalizedEnergyBeamFluxFunction"}:
        if len(blocks) != 1:
            raise ValueError(
                f"Expected one coupled map in {source_definition}; got {len(blocks)}"
            )
        return make_map_source(unit_key, blocks[0], source_definition)

    group_keys = []
    groups = {}
    for block in blocks:
        orientation = tuple(block["Orientation"][-2:])
        beam = tuple(block["Beam"])
        group_key = (beam, orientation)
        if group_key not in groups:
            groups[group_key] = []
            group_keys.append(group_key)
        groups[group_key].append(block)
    if len(groups) != 1:
        raise ValueError(
            f"{source_definition} contains multiple positions/morphologies. "
            "Add an explicit grouping rule before catalog generation."
        )

    (beam, orientation), grouped_blocks = next(iter(groups.items()))
    latitude, longitude = map(float, orientation)
    longitude %= 360.0
    component_names = []
    used_component_names = set()
    spectral_shapes = []
    polarizations = []
    for index, block in enumerate(grouped_blocks, start=1):
        component_name = safe_name(block["_name"])
        if component_name in used_component_names:
            component_name = f"{component_name}_{index}"
        used_component_names.add(component_name)
        component_names.append(component_name)
        spectral_shapes.append(make_parsed_spectrum(block, source_definition))
        polarizations.append(make_polarization(block))

    if beam[0] == "FarFieldPointSource":
        components = [
            SpectralComponent(name, shape, polarization)
            for name, shape, polarization in zip(
                component_names, spectral_shapes, polarizations
            )
        ]
        return PointSource(
            unit_key,
            l=longitude,
            b=latitude,
            components=components,
        )

    if beam[0] == "FarFieldGaussian":
        spatial = Gaussian_on_sphere(
            lon0=longitude,
            lat0=latitude,
            sigma=float(beam[-1]),
        )
    elif beam[0] == "FarFieldAssymetricGaussian":
        axis_a = float(beam[3])
        axis_b = float(beam[4])
        major = max(axis_a, axis_b)
        minor = min(axis_a, axis_b)
        eccentricity = np.sqrt(max(0.0, 1.0 - (minor / major) ** 2))
        spatial = Asymm_Gaussian_on_sphere(
            lon0=longitude,
            lat0=latitude,
            a=major,
            e=eccentricity,
            theta=float(beam[5]),
        )
    else:
        raise NotImplementedError(
            f"Unsupported beam {beam} in {source_definition}"
        )

    # BinnedThreeMLExtendedSourceResponse folds spectrum.main. Collapse
    # co-spatial line/continuum components into one composite spectral shape
    # so every analytic extended source has that standard component path.
    polarization_signatures = [
        None if polarization is None else polarization.to_dict()
        for polarization in polarizations
    ]
    if any(
        signature != polarization_signatures[0]
        for signature in polarization_signatures[1:]
    ):
        raise ValueError(
            f"{source_definition} has co-spatial components with different "
            "polarizations, which cannot be collapsed into one extended source"
        )
    spectral_shape = spectral_shapes[0]
    for component_shape in spectral_shapes[1:]:
        spectral_shape = spectral_shape + component_shape
    return ExtendedSource(
        unit_key,
        spatial_shape=spatial,
        spectral_shape=spectral_shape,
        polarization=polarizations[0],
    )

## Complete DC4 simulation-file manifest

In [26]:
# The first 21 entries use the reviewed continuum models defined above.
BASE_SIMULATION_UNITS = [
    ("cyg_x1_hard", cygx1, "DC3", "cygX1_hard_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("crab", crab, "DC4", "Crab_DC4_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("one_e_1740_compow", one_e_1740, "DC3", "1E1740_compow_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grs_1758_258", grs1758, "DC3", "GRS1758_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("cena", cena, "DC4", "CenA_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("four_c_71p07", four_c_71p07, "DC4", "4C71p07_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("four_c_21p35_noflare", four_c_21p35_noflare, "DC3", "4C21p35_noflare_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("four_c_21p35_flare", four_c_21p35_flare, "DC3", "4C21p35_flare_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("maxi_j1820", maxi_j1820, "DC3", "MAXIJ1820_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("maxi_j1348", maxi_j1348, "DC3", "MAXIJ1348_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("three_c_454p3_low", three_c_454p3_low, "DC4", "3C454p3_low_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("three_c_454p3_high", three_c_454p3_high, "DC4", "3C454p3_high_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("ngc1068", ngc1068, "DC4", "NGC_1068_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("ngc4151", ngc4151, "DC4", "NGC_4151_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("cyg_x3", cyg_x3, "DC4", "cygX3_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("psr_b1259", psr_b1259, "DC3", "PSRB1259_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("one_rxs_j170849", one_rxs_j170849, "DC4", "1RXSJ170849_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("generic_magnetar", generic_magnetar, "DC4", "magnetar2_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("ls_5039", ls_5039, "DC3", "LS5039_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("psr_j1846", psr_j1846, "DC3", "PSRJ1846_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("co_nova_continuum", co_nova_continuum, "DC3", "nova_co_continuum_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
]

# key, source-library challenge, .source stem, event-file challenge, event filename
PARSED_SIMULATION_UNITS = [
    ("tuc_47", "DC3", "Globular_Cluster_Tuc_47", "DC3", "Globular_Cluster_Tuc_47_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("omega_cen", "DC3", "Globular_Cluster_Omega_Cen", "DC3", "Globular_Cluster_Omega_Cen_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("ngc_6397", "DC3", "Globular_Cluster_NGC_6397", "DC3", "Globular_Cluster_NGC_6397_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("ngc_6121", "DC3", "Globular_Cluster_NGC_6121", "DC3", "Globular_Cluster_NGC_6121_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positrons_26al_line", "DC3", "Positrons_from_26Al_line", "DC3", "Positrons_from_26Al_line_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positrons_26al_cont", "DC3", "Positrons_from_26Al_cont", "DC3", "Positrons_from_26Al_cont_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positrons_44ti_line", "DC3", "Positrons_from_44Ti_line", "DC3", "Positrons_from_44Ti_line_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positrons_44ti_cont", "DC3", "Positrons_from_44Ti_cont", "DC3", "Positrons_from_44Ti_cont_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("narrow_bulge_511", "DC3", "NB_511", "DC3", "Narrow_Bulge_511_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("broad_bulge_511", "DC3", "BB_511", "DC3", "Broad_Bulge_511_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("vela_snr_511", "DC3", "Vela_SNR_511", "DC3", "Vela_SNR_511_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("co_nova_511kev", "DC3", "nova_co_511keV", "DC3", "nova_co_511keV_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("co_nova_478kev", "DC3", "nova_co_478keV", "DC3", "nova_co_478keV_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("al26_cyg_region", "DC3", "26Al_Cyg_Region", "DC3", "26Al_Cyg_Region_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("al26_ne2001", "DC3", "26Al_NE2001", "DC3", "26Al_NE2001_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("fe60_cyg_region", "DC3", "60Fe_Cyg_Region", "DC3", "60Fe_Cyg_Region_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("fe60_ne2001", "DC3", "60Fe_NE2001", "DC3", "60Fe_NE2001_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn081207680", "DC3", "bn081207680", "DC3", "GRB_bn081207680_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn090424592", "DC3", "bn090424592", "DC3", "GRB_bn090424592_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn100612726", "DC3", "bn100612726", "DC3", "GRB_bn100612726_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn110605183", "DC3", "bn110605183", "DC3", "GRB_bn110605183_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn131122490", "DC3", "bn131122490", "DC3", "GRB_bn131122490_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn140329295", "DC3", "bn140329295", "DC3", "GRB_bn140329295_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn161004964", "DC3", "bn161004964", "DC3", "GRB_bn161004964_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn170405777", "DC3", "bn170405777", "DC3", "GRB_bn170405777_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn180504136", "DC3", "bn180504136", "DC3", "GRB_bn180504136_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn180703876", "DC3", "bn180703876", "DC3", "GRB_bn180703876_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("grb_bn080802386_flux150", "DC3", "bn080802386", "DC3", "GRB_bn080802386_flux150_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("mgf_051103", "DC3", "MGF051103", "DC3", "GRB_MGF051103_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("mgf_070201", "DC3", "MGF070201", "DC3", "GRB_MGF070201_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("mgf_070222", "DC3", "MGF070222", "DC3", "GRB_MGF070222_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("mgf_180128a", "DC3", "MGF180128A", "DC3", "GRB_MGF180128A_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("mgf_200415a", "DC3", "MGF200415A", "DC3", "GRB_MGF200415A_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("mgf_231115a", "DC3", "MGF231115A", "DC3", "GRB_MGF231115A_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("casa_g16_distribution", "DC4", "CasAG16distribution", "DC4", "CasAG16distribution_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("g1903", "DC4", "G1903", "DC4", "GS1903_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("kepler", "DC4", "Kepler", "DC4", "Kepler_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("mgt_burst_bright_complex", "DC3", "MgtBurst_bright_complex", "DC4", "MgtBurst_bright_complex_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("orion_eridanus", "DC4", "OrEr", "DC4", "OrEr_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positron_annihilation_in_flight", "DC4", "Positrons_In_Flight_Annihilation", "DC4", "positron_annihilation_in_flight_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positrons_central_source", "DC4", "Positrons_Central_Source", "DC4", "Positrons_Central_Source_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positrons_thin_disk_cont", "DC4", "Positrons_Thin_Disk_cont", "DC4", "positrons_thin_disk_cont_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("positrons_thin_disk_line", "DC4", "Positrons_Thin_Disk_line", "DC4", "positrons_thin_disk_line_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("sn1987a", "DC4", "SN1987A", "DC4", "SN1987A_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("tycho", "DC4", "Tycho", "DC4", "Tycho_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("upper_scorpius", "DC4", "USco", "DC4", "USco_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("vela_jr_44ti", "DC4", "VelaJr_44Ti", "DC4", "VelaJr_44Ti_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("lmc_511", "DC3", "LMC_Gaussian_511", "DC4", "LMC_511_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("m31_511", "DC3", "M31_Gaussian_511", "DC4", "M31_511_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
    ("virgo_511", "DC3", "Virgo_Gaussian_511", "DC4", "Virgo_511_3months_unbinned_data_filtered_with_SAAcut.fits.gz"),
]

assert len(BASE_SIMULATION_UNITS) == 21
assert len(PARSED_SIMULATION_UNITS) == 50
assert len(BASE_SIMULATION_UNITS) + len(PARSED_SIMULATION_UNITS) == 71

In [27]:
DC4_ROOT_CANDIDATES = [
    Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files"),
    Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files"),
]
DC4_FILES_ROOT = next(
    (candidate for candidate in DC4_ROOT_CANDIDATES if candidate.exists()),
    None,
)
if DC4_FILES_ROOT is None:
    raise FileNotFoundError(f"Could not find DC4_Files: {DC4_ROOT_CANDIDATES}")
SOURCE_EVENT_ROOT = DC4_FILES_ROOT / "DC4_Sources"

simulation_units = {}
for key, source, event_challenge, event_filename in BASE_SIMULATION_UNITS:
    simulation_units[key] = {
        "source": source,
        "source_definition": "reviewed continuum model",
        "event_file": SOURCE_EVENT_ROOT / event_challenge / event_filename,
    }

for key, model_challenge, source_stem, event_challenge, event_filename in PARSED_SIMULATION_UNITS:
    source_definition = find_source_definition(model_challenge, source_stem)
    simulation_units[key] = {
        "source": parse_megalib_source(key, source_definition),
        "source_definition": str(source_definition),
        "event_file": SOURCE_EVENT_ROOT / event_challenge / event_filename,
    }

if len(simulation_units) != 71:
    raise RuntimeError(f"Expected 71 manifest files, got {len(simulation_units)}")
missing_event_files = [
    str(unit["event_file"])
    for unit in simulation_units.values()
    if not unit["event_file"].exists()
]
if missing_event_files:
    raise FileNotFoundError("Missing native source FITS files:\n" + "\n".join(missing_event_files))

seed_model = Model(*(unit["source"] for unit in simulation_units.values()))
source_types = pd.Series(
    {
        key: type(unit["source"]).__name__
        for key, unit in simulation_units.items()
    },
    name="astromodels type",
)
print("DC4 manifest simulation files:", len(simulation_units))
print("Catalog source entries:", len(seed_model.sources))
display(source_types.value_counts().rename("entries").to_frame())

DC4 manifest simulation files: 71
Catalog source entries: 71


,entries
astromodels type,
PointSource,52
ExtendedSource,19


## NGC 4151 GTI, native-event binning, and mixed responses

Both time selections use the binning from `agn.yaml`.  The full and GTI
extended responses are expensive all-sky response cubes, so they are
cached beside the DC4 files.  Change `OVERWRITE_EXTENDED_RESPONSES` only
when the orientation, GTI, response, or NSIDE changes.

In [28]:
AGN_BINNING_CONFIG = Path(
    "/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/agn.yaml"
)
ORIENTATION_PATH = DC4_FILES_ROOT / "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
RESPONSE_PATH = DC4_FILES_ROOT / (
    "ResponseContinuum.o3.e100_10000.b10log.s10396905069491."
    "m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
)

GTI_TARGET_NAME = "NGC4151"
GTI_MAX_OFFAXIS = 60.0 * u.deg
GTI_EARTH_OCCULTATION = True
LOAD_EXISTING_BINNED_FILES = True
SAVE_BINNED_GTI_FILES = True
OVERWRITE_BINNED_GTI_FILES = False
BINNED_GTI_DIRECTORY = (
    SOURCE_EVENT_ROOT / "DC4_Binned" / f"{GTI_TARGET_NAME}_Cut"
)
NGC4151_READ_ONLY_BINNED_FILE = BINNED_GTI_DIRECTORY / (
    "NGC_4151_ec200_0p30_DC4_COSI_cpl_pl_time_cut_in_fov.hdf5"
)

EXTENDED_RESPONSE_NSIDE = 8
if SPATIAL_TEMPLATE_NSIDE != EXTENDED_RESPONSE_NSIDE:
    raise ValueError(
        "Spatial templates must be area-averaged onto the extended-response NSIDE."
    )
LOAD_EXISTING_EXTENDED_RESPONSES = True
OVERWRITE_EXTENDED_RESPONSES = False
EXTENDED_RESPONSE_DIRECTORY = DC4_FILES_ROOT / "Extended_Responses" / "NGC4151_Cut"
FULL_EXTENDED_RESPONSE_PATH = EXTENDED_RESPONSE_DIRECTORY / (
    f"DC4_continuum_full_extended_response_nside{EXTENDED_RESPONSE_NSIDE}.h5"
)
GTI_EXTENDED_RESPONSE_PATH = EXTENDED_RESPONSE_DIRECTORY / (
    f"DC4_continuum_NGC4151_60deg_GTI_extended_response_nside{EXTENDED_RESPONSE_NSIDE}.h5"
)

for path in (AGN_BINNING_CONFIG, ORIENTATION_PATH, RESPONSE_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

binning_configuration = BinnedData(AGN_BINNING_CONFIG)
measurement_axes = Axes(
    [
        Axis(np.asarray(binning_configuration.energy_bins, dtype=float), unit=u.keV, label="Em"),
        Axis(
            np.linspace(0.0, 180.0, int(180.0 / binning_configuration.phi_pix_size) + 1),
            unit=u.deg,
            label="Phi",
        ),
        HealpixAxis(
            nside=binning_configuration.nside,
            scheme=binning_configuration.scheme,
            coordsys="galactic",
            label="PsiChi",
        ),
    ],
    copy_axes=False,
)
comparison_data = EmCDSBinnedData(Histogram(measurement_axes, sparse=True))

ngc4151_coord = SkyCoord(
    l=ngc4151.position.l.value * u.deg,
    b=ngc4151.position.b.value * u.deg,
    frame="galactic",
)
full_history = SpacecraftHistory.open(ORIENTATION_PATH)
ngc4151_gti = GoodTimeInterval.from_pointing_cut(
    ngc4151_coord,
    full_history,
    GTI_MAX_OFFAXIS,
    earth_occ=GTI_EARTH_OCCULTATION,
)
gti_history = full_history.apply_gti(ngc4151_gti)
detector_response = FullDetectorResponse.open(str(RESPONSE_PATH))

print("Full livetime:", full_history.cumulative_livetime())
print("NGC 4151 GTI livetime:", gti_history.cumulative_livetime())

Full livetime: 6579555.0 s
NGC 4151 GTI livetime: 980415.0 s


In [29]:
def load_or_make_extended_response(history, output_path):
    if output_path.exists() and LOAD_EXISTING_EXTENDED_RESPONSES and not OVERWRITE_EXTENDED_RESPONSES:
        print("Loading cached extended response:", output_path)
        return ExtendedSourceResponse.open(output_path)
    print("Generating extended response (this is expensive):", output_path)
    response = detector_response.get_extended_source_response(
        history,
        coordsys="galactic",
        nside_image=EXTENDED_RESPONSE_NSIDE,
        nside_scatt_map=2 * measurement_axes["PsiChi"].nside,
        earth_occ=GTI_EARTH_OCCULTATION,
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    response.write(output_path, overwrite=OVERWRITE_EXTENDED_RESPONSES)
    return response


def make_mixed_folding(history, extended_response):
    point_response = BinnedThreeMLPointSourceResponse(
        data=comparison_data,
        instrument_response=BinnedInstrumentResponse(detector_response, comparison_data),
        sc_history=history,
        energy_axis=detector_response.axes["Ei"],
        polarization_axis=(
            detector_response.axes["Pol"]
            if "Pol" in detector_response.axes.labels
            else None
        ),
        nside=2 * measurement_axes["PsiChi"].nside,
    )
    extended = BinnedThreeMLExtendedSourceResponse(
        data=comparison_data,
        precomputed_psr=extended_response,
        polarization_axis=(
            extended_response.axes["Pol"]
            if "Pol" in extended_response.axes.labels
            else None
        ),
    )
    return BinnedThreeMLModelFolding(
        data=comparison_data,
        point_source_response=point_response,
        extended_source_response=extended,
    )

In [30]:
def events_in_gti(times, gti):
    times = np.asarray(times, dtype=float)
    starts = np.asarray(gti.tstart_list.unix, dtype=float)
    stops = np.asarray(gti.tstop_list.unix, dtype=float)
    interval_index = np.searchsorted(starts, times, side="right") - 1
    valid = interval_index >= 0
    selected = np.zeros(times.size, dtype=bool)
    selected[valid] = times[valid] < stops[interval_index[valid]]
    return selected


def gti_cache_path(unit_key, event_path):
    if unit_key == "ngc4151" and NGC4151_READ_ONLY_BINNED_FILE.exists():
        return NGC4151_READ_ONLY_BINNED_FILE
    filename = Path(event_path).name
    for suffix in (".fits.gz", ".fits", ".gz"):
        if filename.endswith(suffix):
            filename = filename[: -len(suffix)]
            break
    filename = filename.replace("unbinned_data", "binned_data")
    return BINNED_GTI_DIRECTORY / f"{filename}_{GTI_TARGET_NAME}_Cut.hdf5"


def validate_histogram_axes(histogram, path):
    if tuple(histogram.axes.labels) != tuple(measurement_axes.labels):
        raise ValueError(
            f"{path} axes {histogram.axes.labels} do not match {measurement_axes.labels}"
        )
    expected = tuple(measurement_axes[label].nbins for label in measurement_axes.labels)
    actual = tuple(histogram.axes[label].nbins for label in histogram.axes.labels)
    if actual != expected:
        raise ValueError(f"{path} bin counts {actual} do not match {expected}")
    return histogram


def load_or_bin_native_events(unit_key, event_path):
    cache_path = gti_cache_path(unit_key, event_path)
    cached = (
        LOAD_EXISTING_BINNED_FILES
        and cache_path.exists()
        and not OVERWRITE_BINNED_GTI_FILES
    )
    gti_histogram = (
        validate_histogram_axes(Histogram.open(cache_path), cache_path)
        if cached
        else None
    )

    with fits.open(event_path, memmap=False) as hdul:
        events = hdul[1].data
        energy = np.asarray(events["Energies"], dtype=float)
        phi = np.rad2deg(np.asarray(events["Phi"], dtype=float))
        direction = SkyCoord(
            l=np.asarray(events["Chi galactic"], dtype=float) * u.deg,
            b=np.asarray(events["Psi galactic"], dtype=float) * u.deg,
            frame="galactic",
        )
        full_histogram = Histogram(measurement_axes, sparse=True)
        full_histogram.fill(energy * u.keV, phi * u.deg, direction)

        if gti_histogram is None:
            selected = events_in_gti(events["TimeTags"], ngc4151_gti)
            gti_histogram = Histogram(measurement_axes, sparse=True)
            if np.any(selected):
                gti_histogram.fill(
                    energy[selected] * u.keV,
                    phi[selected] * u.deg,
                    direction[selected],
                )

    if not cached and SAVE_BINNED_GTI_FILES:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if not cache_path.exists() or OVERWRITE_BINNED_GTI_FILES:
            gti_histogram.write(
                cache_path,
                overwrite=OVERWRITE_BINNED_GTI_FILES,
            )
    return full_histogram, gti_histogram, cache_path, "loaded" if cached else "binned"


def load_cached_gti_histogram(unit_key, event_path):
    # The full-response phase creates every missing cache.  Loading only
    # this histogram prevents a second pass through the native FITS files.
    cache_path = gti_cache_path(unit_key, event_path)
    if not cache_path.exists():
        raise FileNotFoundError(
            f"Missing GTI cache {cache_path}. Run the full phase first "
            "with SAVE_BINNED_GTI_FILES=True."
        )
    return validate_histogram_axes(Histogram.open(cache_path), cache_path)


def histogram_counts(histogram):
    return float(histogram.project("Em").to_dense(copy=False).contents.sum())

## Fit and save the two catalogs sequentially

The full response is generated, used, and released before the GTI response
is loaded.  During the full phase every missing GTI histogram is cached.
The GTI phase therefore reads those small HDF5 histograms instead of making
a second pass through the native FITS files.  At NSIDE 8 this keeps only one
approximately 13 GiB extended-response cube resident at a time.

With every spectral and spatial shape fixed, the one-parameter Poisson
maximum-likelihood amplitude is `observed total / forward-folded total`.
Multi-component injected sources are scaled together, preserving their
injected internal component ratios.

Before the expensive response-loading and event-binning loops, every source
is checked with the actual COSI spectral integrators and response energy
bins. Nonfinite, negative, or zero integrated source flux is reported with
the source name. Tabulated spectra use exact piecewise-linear integration.
After updating the COSI integration code, restart the kernel and run this
notebook from the top; rerunning only this cell can retain old imports.
Existing valid response and binned-event caches can still be reused.

In [31]:
CATALOG_DIRECTORY = Path(
    "/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN"
)
manifest_file_count = len(simulation_units)
full_catalog_path = CATALOG_DIRECTORY / (
    f"source_catalog_DC4_all_{manifest_file_count}files_full_fit_norm.yaml"
)
gti_catalog_path = CATALOG_DIRECTORY / (
    f"source_catalog_DC4_all_{manifest_file_count}files_NGC4151_GTI_fit_norm.yaml"
)

if not SAVE_BINNED_GTI_FILES:
    raise ValueError(
        "The sequential workflow requires SAVE_BINNED_GTI_FILES=True so "
        "the GTI phase does not reread every native FITS file."
    )

full_catalog = deepcopy(seed_model)
gti_catalog = deepcopy(seed_model)


def source_amplitudes(source):
    spatial_shape = getattr(source, "spatial_shape", None)
    if isinstance(spatial_shape, GalpropHealpixModel):
        return [spatial_shape.K]
    amplitudes = []
    for component in source.components.values():
        shape_amplitudes = [
            parameter
            for parameter in component.shape.parameters.values()
            if parameter.is_normalization
        ]
        if not shape_amplitudes:
            # Gaussian F is not flagged as a normalization by astromodels.
            # Composite functions rename repeated amplitudes F_1, F_2, ...
            shape_amplitudes = [
                parameter
                for parameter in component.shape.parameters.values()
                if parameter.name.split("_", 1)[0] in {"F", "K", "k"}
            ]
        amplitudes.extend(shape_amplitudes)
    if not amplitudes:
        raise RuntimeError(f"No source amplitude found for {source.name}")
    return amplitudes


def scale_catalog_source(catalog, source_name, scale):
    for amplitude in source_amplitudes(catalog.sources[source_name]):
        target_value = float(amplitude.value) * float(scale)
        if not np.isfinite(target_value) or target_value < 0.0:
            raise ValueError(
                f"Invalid scaled amplitude {target_value} for {source_name}"
            )

        # Several astromodels normalizations default to max_value=1000.
        # Count matching can legitimately require a larger multiplicative K,
        # especially for spatial templates whose initial K is unity. Expand
        # the bound before assignment; this does not change the fitted value.
        if amplitude.max_value is not None and target_value > amplitude.max_value:
            amplitude.max_value = 10.0 ** np.ceil(np.log10(target_value))
        if amplitude.min_value is not None and target_value < amplitude.min_value:
            amplitude.min_value = target_value / 10.0

        amplitude.value = target_value
        set_normalization_step(amplitude)


def freeze_catalog(catalog):
    for parameter in catalog.parameters.values():
        if not parameter.has_auxiliary_variable:
            parameter.fix = True
    assert len(catalog.free_parameters) == 0


def unpolarized_source_copy(source):
    copied_source = deepcopy(source)
    for component in copied_source.components.values():
        degree = getattr(component.polarization, "degree", None)
        if degree is not None:
            degree.value = 0.0
    return copied_source


def folded_source(source, folding):
    source_model = Model(unpolarized_source_copy(source))
    folding.set_model(source_model)
    return folding.expectation()


def fitted_scale(unit_key, observed, expected):
    if not np.isfinite(observed) or observed < 0.0:
        raise ValueError(f"{unit_key}: invalid FITS counts {observed}")
    if not np.isfinite(expected) or expected < 0.0:
        raise ValueError(f"{unit_key}: invalid folded counts {expected}")
    if observed > 0 and expected <= 0:
        raise RuntimeError(
            f"{unit_key}: {observed} FITS counts but zero folded counts"
        )
    if observed <= 0:
        return 1.0e-30
    scale = observed / expected
    if not np.isfinite(scale) or scale <= 0.0:
        raise ValueError(f"{unit_key}: invalid count ratio {scale}")
    return scale


def validate_catalog_sources(units):
    from cosipy.response.functions import (
        get_integrated_spectral_model, get_integrated_extended_model,
    )
    from cosipy.response.functions_3d import get_integrated_extended_model_3d

    energy_axis = detector_response.axes["Ei"]
    image_axis = HealpixAxis(
        nside=EXTENDED_RESPONSE_NSIDE, scheme="ring",
        coordsys="galactic", label="NuLambda",
    )
    issues = []
    for source_name, unit in units.items():
        source = unit["source"]
        if isinstance(source, ExtendedSource) and not hasattr(
            source.spectrum, "main"
        ):
            issues.append(
                f"{source_name}: extended source must use spectrum.main"
            )
        try:
            amplitudes = source_amplitudes(source)
        except Exception as error:
            issues.append(f"{source_name}: {error}")
            continue
        if not amplitudes:
            issues.append(f"{source_name}: no scalable amplitude")
        if isinstance(getattr(source, "spatial_shape", None), GalpropHealpixModel):
            template_path = Path(source.spatial_shape._fitsfile)
            with fits.open(template_path, memmap=False) as hdul:
                template_nside = int(hdul["SKYMAP"].header["NSIDE"])
                template_version = hdul["SKYMAP"].header.get("MAPVERS")
                template_energy = np.asarray(
                    hdul["ENERGIES"].data["ENERGY"], dtype=float
                )
            if template_nside != EXTENDED_RESPONSE_NSIDE:
                issues.append(
                    f"{source_name}: template NSIDE {template_nside} != "
                    f"response NSIDE {EXTENDED_RESPONSE_NSIDE}"
                )
            if template_version != SPATIAL_TEMPLATE_FORMAT_TAG:
                issues.append(
                    f"{source_name}: stale template version {template_version!r}"
                )
            if np.any(np.diff(template_energy) <= 0.0):
                issues.append(
                    f"{source_name}: template energy axis is not increasing"
                )
        # Exercise the actual spectral integrators for EVERY source before
        # loading a large response or reading any native event files.
        # A copy keeps the seed models independent of validation caches.
        try:
            checked_source = unpolarized_source_copy(source)
            if isinstance(checked_source, ExtendedSource):
                integrate_model = (
                    get_integrated_extended_model_3d
                    if isinstance(checked_source.spatial_shape, GalpropHealpixModel)
                    else get_integrated_extended_model
                )
                fluxes = [integrate_model(checked_source, image_axis, energy_axis)]
            else:
                fluxes = [
                    get_integrated_spectral_model(component.shape, energy_axis)
                    for component in checked_source.components.values()
                ]
            total_flux = 0.0
            for flux in fluxes:
                values = np.asarray(flux.contents, dtype=float)
                if not np.all(np.isfinite(values)) or np.any(values < 0.0):
                    raise ValueError(
                        "integrated spectrum contains nonfinite or negative flux"
                    )
                total_flux += float(values.sum())
            if total_flux <= 0.0:
                raise ValueError("zero integrated flux in the response energy band")
        except Exception as error:
            issues.append(f"{source_name}: integration failed: {error}")
    if issues:
        raise RuntimeError(
            "Catalog validation failed before response loading:\n"
            + "\n".join(f"  - {issue}" for issue in issues)
        )
    print(f"Validated {len(units)} foldable catalog sources.")


validate_catalog_sources(simulation_units)


fit_rows = {
    unit_key: {
        "source": unit_key,
        "type": type(unit["source"]).__name__,
    }
    for unit_key, unit in simulation_units.items()
}
cache_rows = []

# -------------------------------------------------------------------------
# Phase 1: full three-month response, native FITS counts, and full YAML.
# Missing GTI histograms are created during this one native-FITS pass.
# -------------------------------------------------------------------------
print("PHASE 1/2: full three-month response and catalog")
full_extended_response = load_or_make_extended_response(
    full_history,
    FULL_EXTENDED_RESPONSE_PATH,
)
full_folding = make_mixed_folding(full_history, full_extended_response)

for unit_index, (unit_key, unit) in enumerate(simulation_units.items(), start=1):
    print(f"[full {unit_index:02d}/{len(simulation_units)}] {unit_key}")
    full_data, gti_data, cache_path, cache_status = load_or_bin_native_events(
        unit_key,
        unit["event_file"],
    )
    full_model_before = folded_source(unit["source"], full_folding)
    full_fits_counts = histogram_counts(full_data)
    full_model_counts = histogram_counts(full_model_before)
    full_scale = fitted_scale(unit_key, full_fits_counts, full_model_counts)
    scale_catalog_source(full_catalog, unit_key, full_scale)

    fit_rows[unit_key].update(
        {
            "full FITS counts": full_fits_counts,
            "full model before": full_model_counts,
            "full scale": full_scale,
            "full model after": full_model_counts * full_scale,
        }
    )
    cache_rows.append(
        {
            "source": unit_key,
            "status": cache_status,
            "GTI cache": str(cache_path),
        }
    )
    del full_data, gti_data, full_model_before

freeze_catalog(full_catalog)
full_catalog.save(full_catalog_path, overwrite=True)
print("Saved full-normalization catalog:", full_catalog_path)

del full_folding, full_extended_response
gc.collect()
print("Released the full extended response before starting the GTI phase.")

# -------------------------------------------------------------------------
# Phase 2: NGC 4151 GTI response, cached GTI counts, and GTI YAML.
# -------------------------------------------------------------------------
print("PHASE 2/2: NGC 4151 GTI response and catalog")
gti_extended_response = load_or_make_extended_response(
    gti_history,
    GTI_EXTENDED_RESPONSE_PATH,
)
gti_folding = make_mixed_folding(gti_history, gti_extended_response)

for unit_index, (unit_key, unit) in enumerate(simulation_units.items(), start=1):
    print(f"[GTI  {unit_index:02d}/{len(simulation_units)}] {unit_key}")
    gti_data = load_cached_gti_histogram(unit_key, unit["event_file"])
    gti_model_before = folded_source(unit["source"], gti_folding)
    gti_fits_counts = histogram_counts(gti_data)
    gti_model_counts = histogram_counts(gti_model_before)
    gti_scale = fitted_scale(unit_key, gti_fits_counts, gti_model_counts)
    scale_catalog_source(gti_catalog, unit_key, gti_scale)

    fit_rows[unit_key].update(
        {
            "GTI FITS counts": gti_fits_counts,
            "GTI model before": gti_model_counts,
            "GTI scale": gti_scale,
            "GTI model after": gti_model_counts * gti_scale,
        }
    )
    del gti_data, gti_model_before

freeze_catalog(gti_catalog)
gti_catalog.save(gti_catalog_path, overwrite=True)
print("Saved NGC 4151 GTI-normalization catalog:", gti_catalog_path)

del gti_folding, gti_extended_response
gc.collect()
print("Released the GTI extended response.")

normalization_table = pd.DataFrame(list(fit_rows.values())).set_index("source")
cache_table = pd.DataFrame(cache_rows).set_index("source")
display(
    normalization_table.style.format(
        {
            "full FITS counts": "{:,.0f}",
            "full model before": "{:,.2f}",
            "full scale": "{:.6g}",
            "full model after": "{:,.2f}",
            "GTI FITS counts": "{:,.0f}",
            "GTI model before": "{:,.2f}",
            "GTI scale": "{:.6g}",
            "GTI model after": "{:,.2f}",
        }
    )
)

Validated 71 foldable catalog sources.
PHASE 1/2: full three-month response and catalog
Loading cached extended response: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/Extended_Responses/NGC4151_Cut/DC4_continuum_full_extended_response_nside8.h5
[full 01/71] cyg_x1_hard
[full 02/71] crab
[full 03/71] one_e_1740_compow
[full 04/71] grs_1758_258
[full 05/71] cena
[full 06/71] four_c_71p07
[full 07/71] four_c_21p35_noflare
[full 08/71] four_c_21p35_flare
[full 09/71] maxi_j1820
[full 10/71] maxi_j1348
[full 11/71] three_c_454p3_low
[full 12/71] three_c_454p3_high
[full 13/71] ngc1068
[full 14/71] ngc4151
[full 15/71] cyg_x3
[full 16/71] psr_b1259
[full 17/71] one_rxs_j170849
[full 18/71] generic_magnetar
[full 19/71] ls_5039
[full 20/71] psr_j1846
[full 21/71] co_nova_continuum
[full 22/71] tuc_47
[full 23/71] omega_cen
[full 24/71] ngc_6397
[full 25/71] ngc_6121
[full 26/71] positrons_26al_line
[full 27/71] positrons_26al_cont
[full 28/71

python(77269) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/768 [00:00<?, ?it/s]

[GTI  01/71] cyg_x1_hard
[GTI  02/71] crab
[GTI  03/71] one_e_1740_compow
[GTI  04/71] grs_1758_258
[GTI  05/71] cena
[GTI  06/71] four_c_71p07
[GTI  07/71] four_c_21p35_noflare
[GTI  08/71] four_c_21p35_flare
[GTI  09/71] maxi_j1820
[GTI  10/71] maxi_j1348
[GTI  11/71] three_c_454p3_low
[GTI  12/71] three_c_454p3_high
[GTI  13/71] ngc1068
[GTI  14/71] ngc4151
[GTI  15/71] cyg_x3
[GTI  16/71] psr_b1259
[GTI  17/71] one_rxs_j170849
[GTI  18/71] generic_magnetar
[GTI  19/71] ls_5039
[GTI  20/71] psr_j1846
[GTI  21/71] co_nova_continuum
[GTI  22/71] tuc_47
[GTI  23/71] omega_cen
[GTI  24/71] ngc_6397
[GTI  25/71] ngc_6121
[GTI  26/71] positrons_26al_line
[GTI  27/71] positrons_26al_cont
[GTI  28/71] positrons_44ti_line
[GTI  29/71] positrons_44ti_cont
[GTI  30/71] narrow_bulge_511
[GTI  31/71] broad_bulge_511
[GTI  32/71] vela_snr_511
[GTI  33/71] co_nova_511kev
[GTI  34/71] co_nova_478kev
[GTI  35/71] al26_cyg_region
[GTI  36/71] al26_ne2001
[GTI  37/71] fe60_cyg_region
[GTI  38/71] fe60

,type,full FITS counts,full model before,full scale,full model after,GTI FITS counts,GTI model before,GTI scale,GTI model after
source,,,,,,,,,
cyg_x1_hard,PointSource,"2,422,239","2,629,484.64",0.921184,"2,422,239.00","10,902","11,662.88",0.93476,"10,902.00"
crab,PointSource,"3,840,085","4,065,295.68",0.944602,"3,840,085.00","599,435","637,409.02",0.940424,"599,435.00"
one_e_1740_compow,PointSource,"218,593","233,696.04",0.935373,"218,593.00","5,934","6,104.15",0.972126,"5,934.00"
grs_1758_258,PointSource,"144,910","162,831.10",0.889941,"144,910.00",908,"1,022.28",0.888214,908.00
cena,PointSource,"113,576","118,380.03",0.959419,"113,576.00","18,066","18,841.74",0.958829,"18,066.00"
four_c_71p07,PointSource,"85,838","88,196.12",0.973263,"85,838.00","36,917","38,945.30",0.947919,"36,917.00"
four_c_21p35_noflare,PointSource,"34,912","45,223.50",0.771988,"34,912.00","19,422","24,711.57",0.785948,"19,422.00"
four_c_21p35_flare,PointSource,"282,694","1,022,114.72",0.276578,"282,694.00","165,892","606,581.88",0.273487,"165,892.00"
maxi_j1820,PointSource,"4,081,328","6,928,341.03",0.589077,"4,081,328.00","45,875","85,686.48",0.535382,"45,875.00"


## Freeze, save, reload, and validate both catalogs

In [32]:
print("Sequential catalogs already saved by the previous cell:")
print("  full:", full_catalog_path)
print("  NGC 4151 GTI:", gti_catalog_path)
print("Peak extended-response residency: one NSIDE-8 cube at a time.")

Sequential catalogs already saved by the previous cell:
  full: /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/source_catalog_DC4_all_71files_full_fit_norm.yaml
  NGC 4151 GTI: /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/source_catalog_DC4_all_71files_NGC4151_GTI_fit_norm.yaml
Peak extended-response residency: one NSIDE-8 cube at a time.


In [33]:
# These custom functions must be imported before load_model in a fresh process.
from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat  # noqa: F401, E402

reloaded_full = load_model(full_catalog_path)
reloaded_gti = load_model(gti_catalog_path)
expected_names = list(simulation_units)
assert len(reloaded_full.sources) == len(expected_names)
assert len(reloaded_gti.sources) == len(expected_names)
assert set(reloaded_full.sources) == set(expected_names)
assert set(reloaded_gti.sources) == set(expected_names)
assert len(reloaded_full.free_parameters) == 0
assert len(reloaded_gti.free_parameters) == 0
print(f"Validated two fixed catalogs with {len(expected_names)} source entries each.")

Validated two fixed catalogs with 71 source entries each.


## Which catalog should the spectral fit load?

- Load `...full_fit_norm.yaml` only with the full three-month
  orientation/data selection.
- Load `...NGC4151_GTI_fit_norm.yaml` with the same NGC 4151 60-degree
  pointing + Earth-occultation GTI used above.
- Do not multiply either catalog by a light curve.  Burst and flare
  durations are already absorbed into their corresponding fitted
  time-averaged amplitudes.
- In the NGC 4151 science fit, remove the frozen catalog copy of
  `ngc4151` before adding the free target model, so the target is not
  counted twice.